In [ ]:
#Importacion de librerias necesarias para el proyecto
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Librerías para visualización
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:

#  Cargar JSON
df = pd.read_json('TelecomX_Data.json')
#df.head()

# Expandir cada columna anidada
customer_expanded = pd.json_normalize(df['customer'])
phone_expanded = pd.json_normalize(df['phone'])
internet_expanded = pd.json_normalize(df['internet'])
account_expanded = pd.json_normalize(df['account'])

# Combinar todo 
df_final = pd.concat([
    df[['customerID', 'Churn']],
    customer_expanded,
    phone_expanded,
    internet_expanded,
    account_expanded
], axis=1)

df_final.head()

In [ ]:
# Verificar si 'charges' está anidado y expandirlo
if 'charges' in df_final.columns:
    charges_expanded = pd.json_normalize(df_final['charges'])
    df_final = pd.concat([df_final.drop('charges', axis=1), charges_expanded], axis=1)
  
df = df_final  

In [ ]:
#  ELIMINAR CUSTOMERID (irrelevante)
df = df.drop('customerID', axis=1)


# LIMPIAR CHURN (eliminar filas con valor vacío)
filas_iniciales = len(df)
df = df[df['Churn'] != ''].copy()

#  LIMPIAR Y RELLENAR CHARGES.TOTAL
# Convertir a numérico (los espacios ' ' se vuelven NaN)
df['Charges.Total'] = pd.to_numeric(df['Charges.Total'], errors='coerce')
nan_count = df['Charges.Total'].isna().sum()
# Rellenar NaN con 0 (clientes nuevos)
df['Charges.Total'] = df['Charges.Total'].fillna(0)


#  TRANSFORMAR VARIABLES BINARIAS (Yes/No → 1/0)
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


#  TRANSFORMAR GENDER (Female/Male → 1/0)
df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})


#  TRANSFORMAR MULTIPLELINES (One-Hot Encoding)
multiple_dummies = pd.get_dummies(df['MultipleLines'], prefix='MultipleLines', drop_first=True)
df = pd.concat([df, multiple_dummies], axis=1)
df = df.drop('MultipleLines', axis=1)


# TRANSFORMAR INTERNETSERVICE (One-Hot Encoding)
internet_dummies = pd.get_dummies(df['InternetService'], prefix='InternetService', drop_first=True)
df = pd.concat([df, internet_dummies], axis=1)
df = df.drop('InternetService', axis=1)

# TRANSFORMAR SERVICIOS DE INTERNET
internet_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                     'TechSupport', 'StreamingTV', 'StreamingMovies']

for service in internet_services:
    dummies = pd.get_dummies(df[service], prefix=service, drop_first=True)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(service, axis=1)


# TRANSFORMAR CONTRACT (One-Hot Encoding)
contract_dummies = pd.get_dummies(df['Contract'], prefix='Contract', drop_first=True)
df = pd.concat([df, contract_dummies], axis=1)
df = df.drop('Contract', axis=1)


# TRANSFORMAR PAYMENTMETHOD (One-Hot Encoding)
payment_dummies = pd.get_dummies(df['PaymentMethod'], prefix='PaymentMethod', drop_first=True)
df = pd.concat([df, payment_dummies], axis=1)
df = df.drop('PaymentMethod', axis=1)


# TRANSFORMAR CHURN (variable objetivo)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


### Verificacion de la proporcion de cancelacion

In [ ]:
# Calcular la proporción de cancelación
churn_proportion = df['Churn'].value_counts(normalize=True) * 100

# Mostrar resultados de forma clara
print(" PROPORCIÓN DE CANCELACIÓN (CHURN):")
print("-" * 40)
print(churn_proportion)
print("-" * 40)

# También podemos ver los conteos absolutos
churn_counts = df['Churn'].value_counts()
print("\n CONTEO ABSOLUTO:")
print(f"Clientes activos (0): {churn_counts[0]}")
print(f"Clientes cancelados (1): {churn_counts[1]}")

### Correlacion

In [ ]:
# PREPARAR DATOS
# Asegurarnos que todas las columnas sean numéricas
df_numerico = df.select_dtypes(include=[np.number])

print(f"\n Dimensiones del dataset numérico: {df_numerico.shape}")
print(f"\n Columnas incluidas ({len(df_numerico.columns)}):")
for i, col in enumerate(df_numerico.columns[:10]):  # Mostrar primeras 10
    print(f"   {i+1}. {col}")
if len(df_numerico.columns) > 10:
    print(f"   ... y {len(df_numerico.columns) - 10} más")

In [ ]:
# CALCULAR MATRIZ DE CORRELACIÓN

correlation_matrix = df_numerico.corr()

#  VISUALIZAR MATRIZ COMPLETA (MAPA DE CALOR)

plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, 
            annot=False,  # Cambiar a True si quieres ver números
            cmap='RdBu_r', 
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

In [ ]:

# ENFOQUE EN CORRELACIÓN CON CHURN 

print("\n" + "=" * 70)
print(" CORRELACIÓN CON LA VARIABLE OBJETIVO (CHURN)")
print("=" * 70)

# Obtener correlaciones con Churn y ordenarlas
corr_con_churn = correlation_matrix['Churn'].drop('Churn').sort_values(ascending=False)

# Convertir a DataFrame para mejor visualización
df_corr_churn = pd.DataFrame({
    'Variable': corr_con_churn.index,
    'Correlación con Churn': corr_con_churn.values,
    '|Correlación|': np.abs(corr_con_churn.values)
}).sort_values('|Correlación|', ascending=False)

print("\n TOP 15 VARIABLES MÁS CORRELACIONADAS CON CHURN:")
print("-" * 70)
print(df_corr_churn.head(15).to_string(index=False))


In [ ]:
# VISUALIZACIÓN GRÁFICA DE CORRELACIONES CON CHURN

plt.figure(figsize=(12, 10))

# Seleccionar top 20 variables (o todas si hay menos)
top_n = min(20, len(df_corr_churn))
top_corr = df_corr_churn.head(top_n)

# Crear barras horizontales
colors = ['crimson' if x > 0 else 'steelblue' for x in top_corr['Correlación con Churn']]
plt.barh(range(top_n), top_corr['Correlación con Churn'], color=colors)
plt.yticks(range(top_n), top_corr['Variable'])
plt.xlabel('Correlación con Churn', fontsize=12)
plt.title(f'Top {top_n} Variables más Correlacionadas con Cancelación (Churn)', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

# ANÁLISIS DE MULTICOLINEALIDAD (VARIABLES REDUNDANTES)

print("\n" + "=" * 70)
print("ANÁLISIS DE MULTICOLINEALIDAD (VARIABLES REDUNDANTES)")
print("=" * 70)

# Encontrar pares con alta correlación (> 0.7)
high_corr_pairs = []
threshold = 0.7

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append({
                'Variable 1': correlation_matrix.columns[i],
                'Variable 2': correlation_matrix.columns[j],
                'Correlación': correlation_matrix.iloc[i, j]
            })

if high_corr_pairs:
    df_high_corr = pd.DataFrame(high_corr_pairs).sort_values('Correlación', ascending=False)
    print(f"\n PARES CON ALTA CORRELACIÓN (> {threshold}):")
    print("-" * 70)
    print(df_high_corr.to_string(index=False))
    
    # Análisis de posibles redundancias
    print("\n RECOMENDACIONES:")
    print("   - Considerar eliminar una variable de cada par redundante")
    print("   - Priorizar mantener la variable con mayor correlación con Churn")
    print("   - Técnicas como PCA pueden ayudar si hay mucha redundancia")
else:
    print(f"\n No se encontraron pares con correlación > {threshold}")



In [ ]:

#  ANÁLISIS ESPECÍFICO DE VARIABLES CLAVE

print("\n" + "=" * 70)
print("🔍 ANÁLISIS DETALLADO DE VARIABLES CLAVE")
print("=" * 70)

# Identificar variables con mayor correlación positiva y negativa
top_positivas = corr_con_churn.head(5)
top_negativas = corr_con_churn.tail(5)

print("\n TOP 5 VARIABLES CON CORRELACIÓN POSITIVA (AUMENTAN CHURN):")
for var, corr in top_positivas.items():
    print(f"   ✅ {var}: +{corr:.3f}")

print("\nTOP 5 VARIABLES CON CORRELACIÓN NEGATIVA (DISMINUYEN CHURN):")
for var, corr in top_negativas.items():
    print(f"   ❌ {var}: {corr:.3f}")